In [1]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
import torch
from qwen_vl_utils import process_vision_info

W0717 19:48:13.993000 48016 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,  # Set False for 8-bit
    bnb_4bit_compute_dtype=torch.float16
)

In [3]:
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-3B-Instruct", torch_dtype="auto", device_map="auto", quantization_config=bnb_config,
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [4]:
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
You have video processor config saved in `preprocessor.json` file which is deprecated. Video processor configs should be saved in their own `video_preprocessor.json` file. You can rename the file or load and save the processor back which renames it automatically. Loading from `preprocessor.json` will be removed in v5.0.


In [ ]:
import os, csv, re
image_folder = './data/train/cloth'
start_from = "05232_00.jpg"
csv_file = 'new_data.csv'
anomaly_file = 'anomaly.csv'
start_processing = False


# Loop through images
for filename in os.listdir(image_folder):
    if filename.lower().endswith(('.jpg')):
        if not start_processing:
            if filename == start_from:
                start_processing = True
                continue
            else:
                continue 
        
        path = os.path.join(image_folder, filename)
        img_name = os.path.basename(path)

        cloth_path = f"./data/train/cloth/{img_name}"
        image_path = f"./data/train/image/{img_name}"

        tagging_prompt = """Please tag the cloth in the image in terms of brand, sleeve, neckline, primary color, secondary color, and casuality.
        Examples: 1.Calvin Klein,long sleeve,v-neck,black,brown,formal 2.Levi's,short sleeve,round neck,blue,white,casual

        Note: If brand name is not present, please use "Unknown" as the brand name. Do not add new schema.
        Please use the following format: [tag1, tag2, tag3, ...].
        Do not include any other text in your response."""

        messages = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "image": cloth_path
                    },
                    {"type": "text", "text": tagging_prompt},
                ],
            }
        ]

        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )
        inputs = inputs.to("cuda")

        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=128)

        generated_ids_trimmed = [
            out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        description_text = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )
        tag_text = description_text[0]

        del inputs, image_inputs, video_inputs, generated_ids, generated_ids_trimmed
        torch.cuda.empty_cache()
        ###############################

        attr_prompt = """Please tag the person's attributes in the image in terms of fit, pants color, and hair color.
        Examples: 1.loose fit,red,black 2.tight fit,black,blonde

        Please use the following format: [tag1, tag2, tag3, ...].
        Do not include any other text in your response."""

        messages = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "image": image_path
                    },
                    {"type": "text", "text": attr_prompt},
                ],
            }
        ]

        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )
        inputs = inputs.to("cuda")


        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=128)

        generated_ids_trimmed = [
            out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        description_text = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )
        attribute_text = description_text[0]

        del inputs, image_inputs, video_inputs, generated_ids, generated_ids_trimmed
        torch.cuda.empty_cache()

        to_strip = "[]"
        tag_text = tag_text.strip(to_strip)
        attribute_text = attribute_text.strip(to_strip)
        data_text = img_name + "," + tag_text + "," + attribute_text
        cleaned_text = re.sub(r'[\[\]]', '', data_text)
        cleaned_text = re.sub(r',\s+', ',', cleaned_text)
        cleaned_text = cleaned_text.lower()

        row_list = cleaned_text.split(',')
        if len(row_list) == 10:
            with open(csv_file, mode='a', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(row_list)
        else:
            with open(anomaly_file, mode='a', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(row_list)


In [ ]:
def get_new_column_value(row):
    img_name = row['img']
    image_path = f"./data/train/image/{img_name}"

    attr_prompt = """Describe the dress in a single line."""

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image_path
                },
                {"type": "text", "text": attr_prompt},
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    inputs = inputs.to("cuda")


    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=128)

    generated_ids_trimmed = [
        out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    description_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    description = description_text[0]
    

    del inputs, image_inputs, video_inputs, generated_ids, generated_ids_trimmed
    torch.cuda.empty_cache()
    
    return description

In [18]:
import csv, os
start_from = "00029_00.jpg"
start_processing = False

input_file = 'new_data.csv'
output_file = 'data_with_new_column.csv'
write_header = not os.path.exists(output_file) or os.path.getsize(output_file) == 0

with open(input_file, 'r', newline='') as infile, open(output_file, 'a', newline='') as outfile:
    reader = csv.DictReader(infile)
    fieldnames = ['img', 'description']  # Only keep these two columns
    writer = csv.DictWriter(outfile, fieldnames=fieldnames)

    if write_header:
        writer.writeheader()

    for row in reader:
        if not start_processing:
            if row['img'] == start_from:
                start_processing = True
                continue
            else:
                continue  

        new_row = {
            'img': row['img'],
            'description': get_new_column_value(row)
        }
        print(row['img'])
        writer.writerow(new_row)


00030_00.jpg
00031_00.jpg
00032_00.jpg
00033_00.jpg
00036_00.jpg
00038_00.jpg
00040_00.jpg
00041_00.jpg
00042_00.jpg
00043_00.jpg
00044_00.jpg
00045_00.jpg
00046_00.jpg
00047_00.jpg
00048_00.jpg
00049_00.jpg
00050_00.jpg
00051_00.jpg
00052_00.jpg
00053_00.jpg
00054_00.jpg
00056_00.jpg
00058_00.jpg
00059_00.jpg
00061_00.jpg
00062_00.jpg
00065_00.jpg
00066_00.jpg
00068_00.jpg
00070_00.jpg
00072_00.jpg
00073_00.jpg
00076_00.jpg
00077_00.jpg
00078_00.jpg
00079_00.jpg
00080_00.jpg
00082_00.jpg
00083_00.jpg
00085_00.jpg
00086_00.jpg
00087_00.jpg
00088_00.jpg
00089_00.jpg
00090_00.jpg
00091_00.jpg
00092_00.jpg
00093_00.jpg
00098_00.jpg
00099_00.jpg
00100_00.jpg
00101_00.jpg
00102_00.jpg
00103_00.jpg
00104_00.jpg
00105_00.jpg
00106_00.jpg
00107_00.jpg
00109_00.jpg
00111_00.jpg
00113_00.jpg
00114_00.jpg
00115_00.jpg
00117_00.jpg
00118_00.jpg
00119_00.jpg
00120_00.jpg
00122_00.jpg
00123_00.jpg
00124_00.jpg
00125_00.jpg
00128_00.jpg
00129_00.jpg
00130_00.jpg
00131_00.jpg
00132_00.jpg
00133_00.jpg

KeyboardInterrupt: 